<a href="https://colab.research.google.com/github/kyungjunoh1/LLM-workspace/blob/main/5_LLama_index.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### LlamaIndex
- 문서를 LLM이 잘 이해할 수 있게 만들어준다
- 문서에서 필요한 정보만 찾는다
- 찾은 정보를 LLM 또는 랭체인과 연동해서 추론한다
### LlamaIndex 순서
- 데이터 저장
  - document객체에 자료를 저장한다
  - 저장된 내용을 chunk_size로 나누어 임베딩처리하여 저장한다
- 추론
  - 질문을 임베딩처리 하여 맞는 문서만 찾아준다
  - 찾은 내용을 LLM에 넘겨 추론한다
    - 질문 + 찾은 문서 -> LLM -> 답변
### 구성 요소 3
1. Documents
  - 라마인덱스에서 사용하는 데이터 단위
2. Index
  - 문서를 검색 가능하게 만드는 구조
    - index = VectorStoreIndex.from_documents(documents)
3. Engine
  - Retriever(문서 검색) : 관련문서(Node)만 가져오는 기능
  - Query Engine(단일 질문) : 검색 + 답변생성( LLM필요 )
    - 이전 대화 내용을 저장하지 않음
  - Chat Engine(대화) : 검색 + 답변(LLM필요) + 대화 기억

In [ ]:
!pip install transformers==4.56.1 -qqq #모델 및 토크나이저
!pip install langchain-huggingface==0.3.1 -qqq #랭체인과 허깅페이스 모델 연결

!pip install llama-index==0.14.8 -qqq
!pip install llama-index-llms-huggingface==0.6.1 -qqq #허깅페이스 모델을 라마인덱스에서 사용
!pip install llama-index-embeddings-huggingface==0.6.1 -qqq #허깅페이스 임베딩 생성
#!pip install llama-index-readers-file==0.5.4 -qqq #파일 읽기

!pip install bitsandbytes==0.47.0 -qqq #성능 최적화

#!pip install hf_xet #로컬pc, 허깅페이스에서 사용하는 고속 파일 다운/저장 시스템

#### Retriever-문서 검색

In [ ]:
from llama_index.core import Settings, Document, VectorStoreIndex
from llama_index.core.llms import ChatMessage, MessageRole
from llama_index.llms.huggingface import HuggingFaceLLM
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

In [ ]:
documents = [
    Document(text="머신러닝은 데이터로부터 패턴을 학습하는 기술이다."),
    Document(text="딥러닝은 머신러닝의 하위 분야이며 인공신경망을 사용한다."),
    Document(text="지도학습은 입력 데이터와 정답 데이터를 함께 학습하는 방식이다."),
    Document(text="사과는 과일이다."),
    Document(text="바나나는 노란색 과일이다."),
    Document(text="포도는 송이로 열리는 과일이다.")
]

In [ ]:
embed_model = HuggingFaceEmbedding( model_name="BM-K/KoSimCSE-roberta" )

In [ ]:
index = VectorStoreIndex.from_documents(documents, embed_model=embed_model)

In [ ]:
ret = index.as_retriever(similarity_top_k = 5)

In [ ]:
nodes = ret.retrieve("과일들 알려줘")
#nodes

In [ ]:
for node in nodes:
  print(node)
  print("-"*20)

In [ ]:
chat_engine = index.as_chat_engine()

In [ ]:
# 라마인덱스 모델 생성
#대화형 모델
#model_id = "kakaocorp/kanana-nano-2.1b-instruct"
#단순 다음 토큰이 어떤값일 지예상하는 모델
model_id = "kakaocorp/kanana-nano-2.1b-base"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    #quantization_config=bnb_config
)

In [ ]:
llm = HuggingFaceLLM(
    model = model,
    tokenizer = tokenizer,
    max_new_tokens=200,
    generate_kwargs={
        "temperature":0.1 # 랜덤하지 않게. 문서 기반
    }
)

In [ ]:
Settings.llm = llm

### Complete
- 단순 질문 답변을 하고자 하는 경우사용(RAG기능 없음)
- RAG기능없이 바로 LLM 추론( 처리속도가 query, chat보다 빠르다)
- 문서 검색 기능이 없기 때문에 prompt로 넣어서 처리해야 한다
### Query
  - 이전 내용을 기억하지 않아 따로 넣어서 사용(대화 유지 chat 보다 부자연스러움)
  - 문서 + LLM 생성(RAG 기능 포함)
  - 문서 없는 경우 LLM 지식으로 답변
### Chat
- 챗봇 기능. 대화형으로 이어가는 경우 사용(자연스럽다)
- 이전 내용을 기억하고 있음(assistant 적용)
- ChatMessage 객체 작성
  - ChatMessage
    - system : 모델 행동 규칙 정의( 너는 기상청 AI야 )
    - user : 사용자 질문 ( 제주도에 장마 언제 시작해? )
    - assistant : 이전 대화( 추론 값 )

In [ ]:
res01 = Settings.llm.complete("머신러닝이 머야")

In [ ]:
res01
res01.text

In [ ]:
res02 = Settings.llm.complete("여기서 말하는 패턴이 뭐야?")
res02.text

In [ ]:
context = res01.text
question = "여기서 말하는 패턴이 뭐야?"
prompt = f"""
너는 머신러닝을 쉽게 설명해주는 AI야
아래 [내용]을 참고해서 대답하고,
모르면 "내용 모름" 으로 대답해

[내용]
{context}

질문:
{question}
"""

In [ ]:
res03 = Settings.llm.complete(prompt)
res03.text

### ChatMessage
- system : 모델 행동 규칙 정의( 너는 기상청 AI야 )
- user : 사용자 질문 ( 제주도에 장마 언제 시작해? )
- assistant : 이전 대화( 추론 값 )

In [ ]:
messages = [
    ChatMessage(role="system", content="너는 머신러닝을 쉽게 설명해주는 AI야"),
    ChatMessage(role="user", content="머신러닝이 뭐야?")
]

In [ ]:
res = Settings.llm.chat(messages)

In [ ]:
res
answer = res.message.content
answer

In [ ]:
messages = [
    ChatMessage(role=MessageRole.SYSTEM, content="너는 머신러닝을 쉽게 설명해주는 AI야"),
    ChatMessage(role=MessageRole.USER, content="비지도 학습은 뭐야?")
]
res = Settings.llm.chat(messages)
res.message.content

In [ ]:
messages.append(ChatMessage(role=MessageRole.ASSISTANT, content = answer))
messages.append(ChatMessage(role=MessageRole.USER, content = "비지도 학습은 뭐야?"))

In [ ]:
len(messages)

In [ ]:
res = Settings.llm.chat(messages)
res.message.content

In [ ]:
len(messages)

In [ ]:
messages[ -2 : ]

In [ ]:
messages[ 0 ]

In [ ]:
[messages[ 0 ]] + messages[ -2 : ]

In [ ]:
documents = [
  Document(text="발주는 최근 7일 평균 판매량을 기준으로 한다.", metadata={"category": "발주"}),
  Document(text="판매량이 증가하면 발주량도 증가시킨다.", metadata={"category": "발주"}),
  Document(text="판매량이 감소하면 발주량을 줄인다.", metadata={"category": "발주"}),
  Document(text="행사 상품은 추가 발주를 고려한다.", metadata={"category": "발주"}),
  Document(text="비 오는 날에는 음료 발주량을 줄인다.", metadata={"category": "발주"}),
  Document(text="더운 날에는 음료 발주량을 늘린다.", metadata={"category": "발주"}),
]

In [ ]:
embed_model

### VectorStoreIndex
- 벡터 DB에 저장될 데이터 및 DB 관리하는 기능
- 기본 vector_store(벡터DB)를 내장하고 있다. 내장 벡터 DB사용 시 휘발성 메모리로 저장된다( 다른 벡터 DB 연동 사용해야 데이터 유지 됨)
- 따로 설정하지 않으면 chunk_size(1024token), chunk_overlap(20token) 설정되어 동작한다(따로 설정 가능)

In [ ]:
index = VectorStoreIndex.from_documents(documents, embed_model=embed_model)

### chat_engine
- 기본 이전 대화 내용을 저장하고 있다.
- 이전 내용 저장하는 용량은 모델의 최대 token과 동일하게 처리된다
  - model.config.max_position_embeddings (최대 초큰 길이)
- chat_mode
  - context
    - 단순 검색용(1회성 검색)
    - 기본 RAG
    - condense_question 보다 빠름
  - condense_question
    - 사용자 질문을 더 잘 검색되게 바꿔서 검색
    - 대화형 챗봇
    - RAG 서비스
    - system_prompt는 LLM설정때 진행해야 한다

In [ ]:
chat_engine = index.as_chat_engine(chat_mode="condense_question")

In [ ]:
simple_question = [
    "발주는 최근 7일 평균 판매량"
    "인기 상품 재고 기준은 무엇인가요?"
    "오늘의 미세먼지는 좋아요?"
]

In [ ]:
for simple in simple_question:
  res = chat_engine.chat(simple)
  print("question : ", simple)
  print()
  print(res.response)
  print("-"*20)

### 유사도 기준으로 추론 설정
- retriever
  - 문서만 검색. llm기능 없음
  - 이전 대화를 기억하지 않으며 단순 일회성 조회때 사용한다
  - 자료 조회용 답변
  - 문서 기반 챗봇
    - retriever + complete (최적)
    - retriever + query (가능)
    - retriever + chat (가능)

In [ ]:
ret = index.as_retriever( similarity_top_k = 3)

In [ ]:
simple_question

In [ ]:
ret.retrieve(simple_question[0])

In [ ]:
nodes = ret.retrieve(simple_question[0])

In [ ]:
for node in nodes:
  print(node)

In [ ]:
nodes = ret.retrieve(simple_question[0])
filter_nodes = [ n for n in nodes if n.score and n.score > 0.7]
filter_nodes

if not filter_nodes:
  print("자료에 없음. 일반 llm 상식으로 추론합니다")
else:
  print(filter_nodes[0].text)

In [ ]:
len(filter_nodes)
filter_nodes[0].text

In [ ]:
content = []
for node in filter_nodes:
  content.append(node.text)
content

In [ ]:
result = "\n\n".join(content)
result

In [ ]:
question = simple_questions[0] #발주는 최근 7일 평균 판매량
prompt = f"""
너는 반드시 아래 자료만 보고 답해야 한다.
자료에 없는 내용은 절대 추측하지 마.

[자료]
{result}

[질문]
{question}

[답변]
"""

In [ ]:
res = Settings.llm.complete(prompt)

In [ ]:
str(res)

In [ ]:
question = simple_question[0] #발주는 최근 7일 평균 판매량
prompt = f"""
너는 반드시 아래 자료만 보고 답해야 한다.
자료에 없는 내용은 절대 추측하지 마.

[자료]
{result}

[답변]
"""

In [ ]:
messages =[
    ChatMessage(role="system", content=prompt),
    ChatMessage(role="user", content=question)
]
res = Settings.llm.chat(messages)
res.message.content

## Ollama 윈도우 설치( 올라마 기능만 확인 후 코랩 다운 실행  )
### Ollama
- 모델 실행 엔진( 하나의 틀 )
- Ollama에 모델을 적용하면 해당 모델로 동작하는 프로그램
### llama3
- 양자화(4bit) 진행된 모델
- 파라미터 8B
- cpu에서도 동작 가능
---
### Ollama 설치
- https://ollama.com/download ( 기본 설치 )
  - microsoft visual c++ 팝업 뜨면 설치 진행
### 실행 환경
- 기본 cli화면이 제공 되며 모델은 기본 gemma3:4b 표현된다
- 질문하면 바로 해당 모델이 다운로드 된다( 질문 : 넌 어떤 모델이야 )
  - 답변 : 저는 Google에서 개발한 대규모 언어 모델인 Gemma입니다. 오픈 웨이트 모델로, 누구나 자유롭게 사용할 수 있습니다.
### cmd창 확인
- ollama --version
- ollama list (아무 내용 없음)
- ollama pull llama3 (올라마 프로그램에서 사용할 모델 다운)
- ollama pull nomic-embed-text (임베딩 모델 다운)
- ollama list
---